# Kitchen Prep Time (KPT) Prediction Model

This notebook trains an XGBoost regression model to predict kitchen prep time based on:
- `queue_length`: Number of orders in the queue
- `orders_last_10min`: Number of orders in the last 10 minutes
- `handover_delay`: Estimated handover delay

Target variable: `actual_prep_time`

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import pickle
import os

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

In [ ]:
# Load dataset
dataset_path = '../data/dataset.csv'
df = pd.read_csv(dataset_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Data exploration
print("Dataset Info:")
print(df.info())
print("\n" + "="*50)
print("\nBasic Statistics:")
print(df.describe())
print("\n" + "="*50)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Prepare features and target
# Features: queue_length, orders_last_10min, handover_delay
features = ['queue_length', 'orders_last_10min', 'handover_delay']
X = df[features]
y = df['actual_prep_time']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature names: {features}")
print(f"\nFeature statistics:")
print(X.describe())

In [ ]:
# Visualize feature distributions and relationships
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Distribution of target variable
axes[0, 0].hist(y, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Actual Prep Time', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Prep Time (minutes)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(y.mean(), color='r', linestyle='--', label=f'Mean: {y.mean():.2f}')
axes[0, 0].legend()

# Queue length vs Prep time
axes[0, 1].scatter(X['queue_length'], y, alpha=0.5, s=20)
axes[0, 1].set_title('Queue Length vs Prep Time', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Queue Length')
axes[0, 1].set_ylabel('Prep Time (minutes)')

# Orders last 10min vs Prep time
axes[1, 0].scatter(X['orders_last_10min'], y, alpha=0.5, s=20, color='green')
axes[1, 0].set_title('Orders Last 10min vs Prep Time', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Orders Last 10min')
axes[1, 0].set_ylabel('Prep Time (minutes)')

# Handover delay vs Prep time
axes[1, 1].scatter(X['handover_delay'], y, alpha=0.5, s=20, color='orange')
axes[1, 1].set_title('Handover Delay vs Prep Time', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Handover Delay (minutes)')
axes[1, 1].set_ylabel('Prep Time (minutes)')

plt.tight_layout()
plt.show()

In [ ]:
# Split dataset into training and testing sets
# Using 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"\nTraining target statistics:")
print(y_train.describe())
print(f"\nTest target statistics:")
print(y_test.describe())

In [ ]:
# Initialize and train XGBoost regression model
print("Training XGBoost model...")

# XGBoost parameters
xgb_params = {
    'objective': 'reg:squarederror',
    'n_estimators': 100,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'n_jobs': -1
}

# Create and train the model
model = xgb.XGBRegressor(**xgb_params)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    eval_metric='mae',
    verbose=False
)

print("Model training completed!")

In [ ]:
# Make predictions on test set
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print("Predictions generated for both training and test sets.")

In [ ]:
# Evaluate model performance
# Calculate MAE (Mean Absolute Error) - primary metric
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)

# Calculate additional metrics for comprehensive evaluation
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

print("="*60)
print("MODEL EVALUATION METRICS")
print("="*60)
print(f"\nTraining Set:")
print(f"  MAE (Mean Absolute Error):  {mae_train:.4f} minutes")
print(f"  RMSE (Root Mean Squared Error): {rmse_train:.4f} minutes")
print(f"  R² Score: {r2_train:.4f}")
print(f"\nTest Set:")
print(f"  MAE (Mean Absolute Error):  {mae_test:.4f} minutes")
print(f"  RMSE (Root Mean Squared Error): {rmse_test:.4f} minutes")
print(f"  R² Score: {r2_test:.4f}")
print("="*60)

In [ ]:
# Plot: Predicted vs Actual Prep Time
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Training set
axes[0].scatter(y_train, y_pred_train, alpha=0.5, s=30)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Prep Time (minutes)', fontsize=12)
axes[0].set_ylabel('Predicted Prep Time (minutes)', fontsize=12)
axes[0].set_title(f'Training Set: Predicted vs Actual\nMAE = {mae_train:.4f} minutes', 
                  fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_pred_test, alpha=0.5, s=30, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Prep Time (minutes)', fontsize=12)
axes[1].set_ylabel('Predicted Prep Time (minutes)', fontsize=12)
axes[1].set_title(f'Test Set: Predicted vs Actual\nMAE = {mae_test:.4f} minutes', 
                  fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot: Residuals analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Residuals for test set
residuals = y_test - y_pred_test

# Residuals distribution
axes[0].hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].axvline(0, color='r', linestyle='--', linewidth=2, label='Zero Error')
axes[0].axvline(residuals.mean(), color='g', linestyle='--', linewidth=2, 
                label=f'Mean: {residuals.mean():.4f}')
axes[0].set_xlabel('Residuals (Actual - Predicted)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Residuals (Test Set)', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals vs Predicted
axes[1].scatter(y_pred_test, residuals, alpha=0.5, s=30, color='orange')
axes[1].axhline(0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Prep Time (minutes)', fontsize=12)
axes[1].set_ylabel('Residuals (Actual - Predicted)', fontsize=12)
axes[1].set_title('Residuals vs Predicted Values (Test Set)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance:")
print(feature_importance)

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('XGBoost Feature Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Save the trained model
model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, 'kpt_model.pkl')

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"Model saved successfully to: {model_path}")
print(f"\nModel file size: {os.path.getsize(model_path) / 1024:.2f} KB")

In [ ]:
# Verify model can be loaded
with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)

# Test prediction with loaded model
sample_features = X_test.iloc[0:1]
sample_prediction = loaded_model.predict(sample_features)
sample_actual = y_test.iloc[0]

print("Model loading verification:")
print(f"Sample features: {sample_features.values[0]}")
print(f"Predicted prep time: {sample_prediction[0]:.4f} minutes")
print(f"Actual prep time: {sample_actual:.4f} minutes")
print(f"Error: {abs(sample_prediction[0] - sample_actual):.4f} minutes")
print("\n✓ Model saved and loaded successfully!")